# Lucrarea de laborator 2

## Procesarea datelor în Databricks, clasificarea recenziilor și vizualizarea în Power BI

### Obiectivul lucrării

Lucrarea continuă procesarea Yelp Open Dataset realizată în laboratorul 1. Se reutilizează datele curățate pentru pregătirea unui subset local de aproximativ 100 MB, care va fi transferat în Databricks.

Fluxul de lucru include:
1. Pregătirea subsetului local.
2. Încărcarea și organizarea datelor în Databricks.
3. Antrenarea și evaluarea unui model de clasificare cu Spark MLlib.
4. Salvarea predicțiilor și demonstrarea mecanismelor de fiabilitate.
5. Construirea unui raport interactiv în Power BI.

### Problema predictivă

Pe baza textului unei recenzii se urmărește identificarea recenziilor cu rating scăzut:
- `label = 1`: rating de 1–2 stele;
- `label = 0`: rating de 3–5 stele.

Clasa `0` înseamnă „fără rating scăzut”, nu neapărat sentiment pozitiv.

Textul reprezintă intrarea modelului. Ratingul este utilizat pentru construirea etichetei de referință și nu va fi inclus ca variabilă predictivă.

## 1. Pregătirea subsetului local

### 1.1. Sursa datelor

Se utilizează tabelul Delta rezultat din etapa de prelucrare a laboratorului 1:

`laborator_1/data/processed_delta`

Tabelul conține recenziile și contextul asociat afacerilor. Schema sa păstrează informațiile necesare atât clasificării, cât și raportului Power BI:

| Coloane | Utilizare |
|---|---|
| `review_id` | Identificarea recenziei și verificarea unicității |
| `business_id` | Identificarea afacerii |
| `text` | Intrarea modelului de clasificare |
| `stars` | Construirea etichetei |
| `date`, `year`, `month` | Analiza temporală |
| `name`, `city`, `state`, `main_category` | Contextul afacerii și filtrele raportului |
| `text_length`, `useful`, `funny`, `cool`, `total_votes` | Atribute disponibile pentru analiza descriptivă |

Datele despre afaceri sunt deja incluse în sursă, astfel încât nu este necesară o asociere suplimentară cu `dim_business`.

Fluxul de curățare din laboratorul 1 nu este repetat. În această etapă se verifică datele de intrare, se pregătește o selecție reproductibilă și se exportă subsetul în format Parquet.

Ținta de aproximativ 100 MB se referă la dimensiunea efectivă a fișierelor Parquet exportate.

### 1.2. Inițializarea mediului local

Pregătirea subsetului se execută în mediul local utilizat pentru laboratorul 1.

Se reutilizează funcția `get_spark` din modulul `src.lab_common`, cu integrarea Kafka dezactivată.

Căile sursei și destinației sunt definite explicit. Tabelul Delta din laboratorul 1 este utilizat numai pentru citire; subsetul va fi salvat separat, în directorul laboratorului 2.

In [1]:
from pathlib import Path
import sys

from pyspark.sql import functions as F

# Configurația locală a repository-ului.
REPO_ROOT = Path("/home/ivan/tmp/new/big_data")

LAB1_ROOT = REPO_ROOT / "laborator_1"
SOURCE_PATH = LAB1_ROOT / "data" / "processed_delta"
SUBSET_PATH = REPO_ROOT / "laborator_2" / "data" / "yelp_subset"

if not (SOURCE_PATH / "_delta_log").is_dir():
    raise FileNotFoundError(
        f"Tabelul Delta nu este disponibil la calea: {SOURCE_PATH}"
    )

if str(LAB1_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB1_ROOT))

from src.lab_common import get_spark

spark = get_spark(app="yelp-lab2-subset", kafka=False)

reviews_clean = spark.read.format("delta").load(str(SOURCE_PATH))

print("Spark:", spark.version)
print("Sursă:", SOURCE_PATH)
print("Destinația planificată:", SUBSET_PATH)

:: loading settings :: url = jar:file:/home/ivan/.mamba/envs/Playground/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ivan/.ivy2.5.2/cache
The jars for the packages stored in: /home/ivan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b850eb39-29e3-4033-a178-7c16fe3070df;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-

Spark: 4.1.1
Sursă: /home/ivan/tmp/new/big_data/laborator_1/data/processed_delta
Destinația planificată: /home/ivan/tmp/new/big_data/laborator_2/data/yelp_subset


### 1.3. Verificarea datelor de intrare

Înaintea selecției se execută verificări de control asupra sursei:
- existența coloanelor necesare;
- completitudinea și unicitatea identificatorilor recenziilor;
- prezența unui text nevid;
- apartenența ratingului la valorile 1, 2, 3, 4 sau 5;
- prezența datei și concordanța acesteia cu anul;
- disponibilitatea contextului utilizat în dashboard.

Aceste verificări nu modifică tabelul sursă. Eventualele probleme vor fi analizate înainte de stabilirea regulilor de selecție.

Agregările sunt executate pe întregul tabel și presupun citirea datelor, nu doar consultarea metadatelor.

In [2]:
required_columns = [
    "review_id",
    "business_id",
    "text",
    "stars",
    "date",
    "year",
    "month",
    "name",
    "city",
    "state",
    "main_category",
    "text_length",
    "useful",
    "funny",
    "cool",
    "total_votes",
]

missing_columns = sorted(set(required_columns) - set(reviews_clean.columns))

if missing_columns:
    raise ValueError(f"Coloane necesare lipsă: {missing_columns}")


def is_missing_text(column_name):
    column = F.col(column_name)
    return column.isNull() | (F.length(F.trim(column)) == 0)


def count_if(condition, alias):
    return F.count(F.when(condition, F.lit(1))).alias(alias)


invalid_id = is_missing_text("review_id")
invalid_text = is_missing_text("text")

invalid_stars = (
    F.col("stars").isNull()
    | F.isnan("stars")
    | ~F.col("stars").isin(1.0, 2.0, 3.0, 4.0, 5.0)
)

invalid_date = F.col("date").isNull()

invalid_year = (
    F.col("year").isNull()
    | (F.col("year") != F.year("date"))
)

quality = reviews_clean.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("review_id").alias("n_distinct_review_ids"),
    count_if(invalid_id, "invalid_review_id"),
    count_if(invalid_text, "invalid_text"),
    count_if(invalid_stars, "invalid_stars"),
    count_if(invalid_date, "invalid_date"),
    count_if(invalid_year, "invalid_year"),
    count_if(is_missing_text("business_id"), "missing_business_id"),
    count_if(is_missing_text("name"), "missing_business_name"),
    count_if(is_missing_text("city"), "missing_city"),
    count_if(is_missing_text("state"), "missing_state"),
    count_if(is_missing_text("main_category"), "missing_main_category"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.round(F.avg(F.length("text")), 1).alias("avg_text_characters"),
).first().asDict()

for key, value in quality.items():
    print(f"{key}: {value}")

unique_valid_ids = (
    quality["invalid_review_id"] == 0
    and quality["n_rows"] == quality["n_distinct_review_ids"]
)

print("\nIdentificatori review_id unici și nenuli:", unique_valid_ids)

26/10/27 09:18:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


n_rows: 6958470
n_distinct_review_ids: 6958470
invalid_review_id: 0
invalid_text: 0
invalid_stars: 0
invalid_date: 0
invalid_year: 0
missing_business_id: 0
missing_business_name: 0
missing_city: 0
missing_state: 0
missing_main_category: 0
min_date: 2005-02-16 03:23:22
max_date: 2021-12-31 23:59:26
avg_text_characters: 568.0

Identificatori review_id unici și nenuli: True


### 1.4. Distribuția temporală și distribuția claselor

Se examinează numărul recenziilor pe ani și distribuția ratingurilor pentru a caracteriza sursa înainte de eșantionare.

Pentru analiza claselor se construiește explicit eticheta `label` din coloana `stars`:
- `1` pentru recenziile cu 1–2 stele;
- `0` pentru recenziile cu 3–5 stele.

Coloana existentă `is_positive` nu este reutilizată, deoarece definiția ei nu este presupusă identică problemei de clasificare din această lucrare.

Distribuția pe ani permite evaluarea acoperirii temporale, iar proporția clasei cu rating scăzut evidențiază un eventual dezechilibru între clase.

Analiza etichetelor include numai ratingurile valide. Tabelul sursă nu este modificat.

In [3]:
reviews_distribution = (
    reviews_clean
    .filter(~invalid_stars)
    .withColumn(
        "label",
        F.when(F.col("stars") <= 2.0, F.lit(1)).otherwise(F.lit(0)),
    )
)

year_distribution = (
    reviews_distribution
    .groupBy("year")
    .agg(
        F.count("*").alias("n_reviews"),
        F.sum("label").alias("n_low_rating"),
        F.round(
            100.0 * F.avg("label"), 2
        ).alias("low_rating_pct"),
    )
    .orderBy("year")
)

print("Distribuția pe ani — ratinguri valide:")
year_distribution.show(100, truncate=False)

print("Distribuția pe rating — sursa completă:")
(
    reviews_clean
    .groupBy("stars")
    .count()
    .orderBy("stars")
    .show(truncate=False)
)

print("Distribuția claselor — ratinguri valide:")
(
    reviews_distribution
    .groupBy("label")
    .count()
    .orderBy("label")
    .show(truncate=False)
)

Distribuția pe ani — ratinguri valide:
+----+---------+------------+--------------+
|year|n_reviews|n_low_rating|low_rating_pct|
+----+---------+------------+--------------+
|2005|854      |84          |9.84          |
|2006|3853     |351         |9.11          |
|2007|15363    |1855        |12.07         |
|2008|48224    |6338        |13.14         |
|2009|74386    |12623       |16.97         |
|2010|138584   |23830       |17.2          |
|2011|230809   |42823       |18.55         |
|2012|286569   |58346       |20.36         |
|2013|383942   |81416       |21.21         |
|2014|522271   |113799      |21.79         |
|2015|688412   |154894      |22.5          |
|2016|758869   |173982      |22.93         |
|2017|820033   |189115      |23.06         |
|2018|906346   |214443      |23.66         |
|2019|907245   |221958      |24.47         |
|2020|554548   |140400      |25.32         |
|2021|618162   |168934      |27.33         |
+----+---------+------------+--------------+

Distribuția pe 

### 1.5. Rezultatele verificării sursei

Tabelul sursă conține **6.958.470 de recenzii**, fiecare având un identificator `review_id` unic și nenul.

Controalele executate nu au identificat:
- texte nule sau vide;
- ratinguri lipsă sau în afara valorilor permise;
- date lipsă sau neconcordanțe între dată și an;
- valori lipsă în câmpurile de context verificate: identificatorul și denumirea afacerii, orașul, statul și categoria principală.

Recenziile acoperă intervalul **16 februarie 2005 – 31 decembrie 2021**, iar lungimea medie a textului este de **568 de caractere**.

Distribuția claselor este dezechilibrată:
- clasa `0`, fără rating scăzut: **5.353.279 de recenzii**;
- clasa `1`, cu rating scăzut: **1.605.191 de recenzii**.

Proporția recenziilor cu rating scăzut diferă între ani: de exemplu, este de 9,84% în 2005 și de 27,33% în 2021. Această variație justifică păstrarea dimensiunii temporale pentru analiza ulterioară.

Rezultatele permit continuarea pregătirii subsetului fără aplicarea unor corecții suplimentare pentru problemele verificate. Ele nu demonstrează absența tuturor problemelor semantice posibile, precum variațiile de scriere ale orașelor.

### 1.6. Selecția reproductibilă a recenziilor

Subsetul este construit printr-o selecție deterministă bazată pe hash-ul identificatorului `review_id`.

Fiecărei recenzii îi este atribuit un bucket între 0 și 9.999. Sunt păstrate recenziile al căror bucket este mai mic decât pragul configurat.

Metoda oferă:
- repetabilitatea selecției pentru aceeași sursă și aceeași configurație;
- independență față de ordinea fizică a rândurilor;
- posibilitatea de a ajusta proporția fără schimbarea metodei;
- subseturi imbricate: creșterea pragului păstrează recenziile deja selectate.

Selecția este aplicată întregului interval temporal, fără echilibrarea artificială a claselor. Se urmărește păstrarea aproximativă a distribuțiilor sursei, nu un număr identic de recenzii pentru fiecare an sau clasă.

Prima selecție utilizează un prag de 500 din 10.000, corespunzător unei proporții nominale de 5%. Numărul efectiv de rânduri și dimensiunea fișierelor sunt măsurate după export.

Coloanele exportate includ textul, eticheta și contextul necesar analizei. Prezența ratingului și a contextului în subset nu înseamnă că acestea vor fi utilizate ca intrări ale modelului: pentru clasificarea propusă, intrarea predictivă rămâne textul.

In [4]:
# Parametrii selecției.
HASH_BUCKETS = 10_000
SELECTION_THRESHOLD = 500  # Proporție nominală: 5%.

assert 1 <= SELECTION_THRESHOLD <= HASH_BUCKETS

export_columns = [
    "review_id",
    "business_id",
    "text",
    "stars",
    "date",
    "year",
    "month",
    "name",
    "city",
    "state",
    "main_category",
    "label",
]

reviews_labeled = reviews_clean.withColumn(
    "label",
    F.when(F.col("stars") <= 2.0, F.lit(1.0)).otherwise(F.lit(0.0)),
)

subset = (
    reviews_labeled
    .withColumn(
        "_selection_bucket",
        F.pmod(F.xxhash64("review_id"), F.lit(HASH_BUCKETS)),
    )
    .filter(F.col("_selection_bucket") < SELECTION_THRESHOLD)
    .select(*export_columns)
)

print(
    "Proporție nominală selectată:",
    f"{100 * SELECTION_THRESHOLD / HASH_BUCKETS:.2f}%",
)

subset.printSchema()

Proporție nominală selectată: 5.00%
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- label: double (nullable = false)



### 1.7. Exportul local în Parquet

Subsetul este salvat în directorul:

`laborator_2/data/yelp_subset`

Se utilizează formatul Parquet cu compresie Snappy. Pentru acest subset nu se aplică partiționarea pe ani, pentru a evita fragmentarea în numeroase fișiere mici, în special pentru anii cu puține recenzii.

Exportul este distribuit în patru partiții de execuție. Numărul de partiții nu garantează o anumită dimensiune a fișierelor.

Modul `overwrite` permite regenerarea exportului la ajustarea pragului de selecție. Acesta înlocuiește exclusiv conținutul destinației subsetului și nu modifică tabelul sursă.

După export se măsoară:
- dimensiunea cumulată a fișierelor Parquet;
- dimensiunea totală a directorului, inclusiv fișierele auxiliare;
- numărul de recenzii exportate;
- distribuția claselor și acoperirea temporală.

În această lucrare, ținta de 100 MB este interpretată în unități zecimale: 100.000.000 de octeți. Dimensiunea în MiB este afișată separat.

Dimensiunea efectivă depinde de lungimea textelor, coloanele păstrate și compresie; de aceea pragul de selecție este calibrat pe baza exportului măsurat.

In [5]:
TARGET_BYTES = 100_000_000
EXPORT_PARTITIONS = 4

# Protecție împotriva suprascrierii accidentale a altei destinații.
expected_subset_path = (
    REPO_ROOT / "laborator_2" / "data" / "yelp_subset"
).resolve()

assert SUBSET_PATH.resolve() == expected_subset_path
assert SOURCE_PATH.resolve() != SUBSET_PATH.resolve()

(
    subset
    .repartition(EXPORT_PARTITIONS)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(str(SUBSET_PATH))
)

# Verificăm datele citite din export, nu doar DataFrame-ul de dinaintea scrierii.
exported_subset = spark.read.parquet(str(SUBSET_PATH))

parquet_files = sorted(SUBSET_PATH.rglob("*.parquet"))
all_files = [
    path for path in SUBSET_PATH.rglob("*")
    if path.is_file()
]

parquet_bytes = sum(path.stat().st_size for path in parquet_files)
directory_bytes = sum(path.stat().st_size for path in all_files)

if parquet_bytes == 0:
    raise RuntimeError("Exportul nu conține fișiere Parquet cu date.")

export_summary = exported_subset.agg(
    F.count("*").alias("n_reviews"),
    F.countDistinct("review_id").alias("n_distinct_review_ids"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
).first().asDict()

print("Destinație:", SUBSET_PATH)
print("Fișiere Parquet:", len(parquet_files))
print(f"Dimensiune Parquet: {parquet_bytes / 1_000_000:.2f} MB")
print(f"Dimensiune Parquet: {parquet_bytes / (1024 ** 2):.2f} MiB")
print(f"Dimensiune totală director: {directory_bytes / 1_000_000:.2f} MB")
print(
    "Proporție efectivă din rândurile sursei:",
    f"{100 * export_summary['n_reviews'] / quality['n_rows']:.2f}%",
)

for key, value in export_summary.items():
    print(f"{key}: {value}")

print("\nDistribuția claselor în export:")
(
    exported_subset
    .groupBy("label")
    .agg(
        F.count("*").alias("n_reviews"),
    )
    .withColumn(
        "percentage",
        F.round(
            100.0 * F.col("n_reviews") / F.lit(export_summary["n_reviews"]),
            2,
        ),
    )
    .orderBy("label")
    .show(truncate=False)
)

print("Distribuția temporală în export:")
(
    exported_subset
    .groupBy("year")
    .count()
    .orderBy("year")
    .show(100, truncate=False)
)

# Estimare pentru următoarea rulare, fără reexport automat.
suggested_threshold = max(
    1,
    min(
        HASH_BUCKETS,
        round(SELECTION_THRESHOLD * TARGET_BYTES / parquet_bytes),
    ),
)

print(
    "\nPrag estimat pentru apropierea de 100 MB:",
    suggested_threshold,
)
print(
    "Estimarea presupune o relație aproximativ proporțională "
    "între prag și dimensiunea exportului."
)

Destinație: /home/ivan/tmp/new/big_data/laborator_2/data/yelp_subset
Fișiere Parquet: 4
Dimensiune Parquet: 148.78 MB
Dimensiune Parquet: 141.89 MiB
Dimensiune totală director: 149.94 MB
Proporție efectivă din rândurile sursei: 5.00%
n_reviews: 347694
n_distinct_review_ids: 347694
min_date: 2005-03-01 16:59:37
max_date: 2021-12-31 23:50:20

Distribuția claselor în export:
+-----+---------+----------+
|label|n_reviews|percentage|
+-----+---------+----------+
|0.0  |267449   |76.92     |
|1.0  |80245    |23.08     |
+-----+---------+----------+

Distribuția temporală în export:
+----+-----+
|year|count|
+----+-----+
|2005|47   |
|2006|185  |
|2007|768  |
|2008|2550 |
|2009|3672 |
|2010|6911 |
|2011|11480|
|2012|14506|
|2013|19053|
|2014|26270|
|2015|34073|
|2016|37794|
|2017|41085|
|2018|45142|
|2019|45511|
|2020|27710|
|2021|30937|
+----+-----+


Prag estimat pentru apropierea de 100 MB: 336
Estimarea presupune o relație aproximativ proporțională între prag și dimensiunea exportului.


### 1.8. Rezultatul pregătirii subsetului local

Subsetul final a fost exportat în directorul `laborator_2/data/yelp_subset`, în format Parquet cu compresie Snappy.

| Indicator | Rezultat |
|---|---:|
| Număr de recenzii | 347.694 |
| Identificatori `review_id` distincți | 347.694 |
| Proporție efectivă din sursă, rotunjită | 5,00% |
| Număr de fișiere Parquet | 4 |
| Dimensiunea cumulată a fișierelor Parquet | 148,78 MB |
| Dimensiunea totală a directorului | 149,94 MB |
| Prima recenzie | 2005-03-01 16:59:37 |
| Ultima recenzie | 2021-12-31 23:50:20 |

Deși exportul depășește ținta inițială de aproximativ 100 MB, se păstrează această variantă pentru continuarea lucrării.

#### Distribuția claselor

| Clasă | Semnificație | Recenzii | Proporție |
|---|---|---:|---:|
| `0.0` | Rating de 3–5 stele | 267.449 | 76,92% |
| `1.0` | Rating de 1–2 stele | 80.245 | 23,08% |

Subsetul păstrează ambele clase și recenzii din fiecare an al intervalului 2005–2021. Clasele nu au fost echilibrate artificial.

Numărul identificatorilor distincți este egal cu numărul de rânduri, ceea ce confirmă absența duplicatelor după `review_id` în export.

Selecția este reproductibilă pentru aceeași sursă și aceeași configurație, fiind bazată pe hash-ul identificatorului recenziei, cu un prag de 500 din 10.000 de bucket-uri.

Subsetul exportat constituie intrarea pentru etapa de încărcare în Databricks.